# AAI614: Data Science & its Applications

*Notebook 3.2: Practice with Data Cleaning*

<a href="https://colab.research.google.com/github/harmanani/AAI614/blob/main/Week%203/Notebook3.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

Exercise I. Load the following datafile from GitHub

In [2]:
grads = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")

In [3]:
grads

,Student Name,Avg Hours Studies per Week,GPA,University,Sense of Humour (0-5),Salary
0,George,20,NaN,NYU,3.0,$40k
1,Jerry,35,3.5,Columbia,5.0,$80k
2,Elaine,55,4.0,Columbia,4.2,$60k
3,Cosmo,5,2.0,City College,2.0,$25k
4,Newman,25,2.8,City College,0.0,$50k
5,Frank,35,3.0,Festivus Uni,NaN,$40k
6,Estelle,100,3.2,Festivus Uni,1.7,$0k
7,Leo,15,2.4,Festivus Uni,0.0,$35k
8,Rachel,50,4.0,Columbia,NaN,$75k


Question 1: Identify all the outliers in the above data.  Justify your answers using objective measures.

In [6]:
import numpy as np                       # tools for numbers (needed later)

grads.columns = ["Name", "Hours", "GPA", "University", "Humour", "Salary"]
# gives the columns short names, so the code below is easier to write

grads["Salary"] = grads["Salary"].str.replace("$", "", regex=False)
# removes the "$" sign, so "$40k" becomes "40k"

grads["Salary"] = grads["Salary"].str.replace("k", "", regex=False).astype(float)
# removes the "k" and turns the text "40" into the number 40.0

grads
# shows the table so you can check it. Salary should now be numbers (40.0, 80.0...)

,Name,Hours,GPA,University,Humour,Salary
0,George,20,NaN,NYU,3.0,40.0
1,Jerry,35,3.5,Columbia,5.0,80.0
2,Elaine,55,4.0,Columbia,4.2,60.0
3,Cosmo,5,2.0,City College,2.0,25.0
4,Newman,25,2.8,City College,0.0,50.0
5,Frank,35,3.0,Festivus Uni,NaN,40.0
6,Estelle,100,3.2,Festivus Uni,1.7,0.0
7,Leo,15,2.4,Festivus Uni,0.0,35.0
8,Rachel,50,4.0,Columbia,NaN,75.0


In [8]:
for col in ["Hours", "GPA", "Humour", "Salary"]:
# repeats everything below once for each of these four columns

    q1 = grads[col].quantile(0.25)      # Q1: the value 25% of the way up the data
    q3 = grads[col].quantile(0.75)      # Q3: the value 75% of the way up the data
    iqr = q3 - q1                       # IQR: the spread of the middle half
    lower = q1 - 1.5 * iqr              # below this = outlier
    upper = q3 + 1.5 * iqr              # above this = outlier

    z = (grads[col] - grads[col].mean()) / grads[col].std()
    # z-score: how many standard deviations each value is from the average

    print(col, "| lower fence:", lower, "| upper fence:", upper)
    # prints the column name and its two limits

    print("IQR outliers:", grads.loc[(grads[col] < lower) | (grads[col] > upper), "Name"].tolist())
    # finds names whose value is below the lower or above the upper limit

    print("|z| > 2:", grads.loc[z.abs() > 2, "Name"].tolist(), "\n")
    # finds names whose z-score is more than 2 away from 0 (either direction)
    z_hours = (grads["Hours"] - grads["Hours"].mean()) / grads["Hours"].std()
print(z_hours.round(2))

Hours | lower fence: -25.0 | upper fence: 95.0
IQR outliers: ['Estelle']
|z| > 2: ['Estelle'] 

GPA | lower fence: 1.3124999999999993 | upper fence: 5.0125
IQR outliers: []
|z| > 2: [] 

Humour | lower fence: -3.275 | upper fence: 7.725
IQR outliers: []
|z| > 2: [] 

Salary | lower fence: -2.5 | upper fence: 97.5
IQR outliers: []
|z| > 2: [] 

0   -0.63
1   -0.10
2    0.61
3   -1.16
4   -0.45
5   -0.10
6    2.20
7   -0.80
8    0.43
Name: Hours, dtype: float64


Estelle's 100 hours of study per week is the only outlier. Its z-score is 2.20, so it is more than 2 standard deviations from the mean, while all other z-scores are between -1.16 and 0.61. It is also above the IQR upper fence of 95. No value in GPA, Sense of Humour or Salary is outside its fences or has a z-score above 2.

Question 2: There are various data that are missing.  Fill-in the missing data or delete the rows/columns that you think you should delete.  Justify your answer

In [9]:
print(grads.isnull().sum())

grads["GPA"] = grads["GPA"].fillna(grads["GPA"].median())
grads["Humour"] = grads["Humour"].fillna(grads["Humour"].median())

print(grads.isnull().sum())
grads

Name          0
Hours         0
GPA           1
University    0
Humour        2
Salary        0
dtype: int64
Name          0
Hours         0
GPA           0
University    0
Humour        0
Salary        0
dtype: int64


,Name,Hours,GPA,University,Humour,Salary
0,George,20,3.1,NYU,3.0,40.0
1,Jerry,35,3.5,Columbia,5.0,80.0
2,Elaine,55,4.0,Columbia,4.2,60.0
3,Cosmo,5,2.0,City College,2.0,25.0
4,Newman,25,2.8,City College,0.0,50.0
5,Frank,35,3.0,Festivus Uni,2.0,40.0
6,Estelle,100,3.2,Festivus Uni,1.7,0.0
7,Leo,15,2.4,Festivus Uni,0.0,35.0
8,Rachel,50,4.0,Columbia,2.0,75.0


Three values are missing: one GPA (George) and two Sense of Humour values (Frank and Rachel). I did not delete any rows or columns. Each row is missing at most one value out of six, so the ratio of missing elements is low, and deleting rows would remove a large part of a dataset of only 9 students. Missing values can reduce model performance and add bias, so I filled them instead. GPA and Sense of Humour are continuous numeric attributes, so substitution with a summary value is suitable. I used the median because the lesson says it is more robust than the average, being less sensitive to extreme values. The missing GPA was filled with 3.1 and the two missing Sense of Humour values with 2.0.

Question 3: Reload the data and fill-in the data using mean method as well as the frequent method.

In [10]:
raw = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")
# reloads the original data, with the missing values back

raw.columns = ["Name", "Hours", "GPA", "University", "Humour", "Salary"]
# same short names as before

df_mean = raw.copy()     # copy to fill with the mean
df_freq = raw.copy()     # copy to fill with the most frequent value

for col in ["GPA", "Humour"]:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())
    # replaces missing values with the average of the column

    df_freq[col] = df_freq[col].fillna(df_freq[col].mode()[0])
    # replaces missing values with the most frequent value of the column

print("Mean method:")
print(df_mean[["Name", "GPA", "Humour"]])
print("\nFrequent method:")
print(df_freq[["Name", "GPA", "Humour"]])

Mean method:
      Name     GPA    Humour
0   George  3.1125  3.000000
1    Jerry  3.5000  5.000000
2   Elaine  4.0000  4.200000
3    Cosmo  2.0000  2.000000
4   Newman  2.8000  0.000000
5    Frank  3.0000  2.271429
6  Estelle  3.2000  1.700000
7      Leo  2.4000  0.000000
8   Rachel  4.0000  2.271429

Frequent method:
      Name  GPA  Humour
0   George  4.0     3.0
1    Jerry  3.5     5.0
2   Elaine  4.0     4.2
3    Cosmo  2.0     2.0
4   Newman  2.8     0.0
5    Frank  3.0     0.0
6  Estelle  3.2     1.7
7      Leo  2.4     0.0
8   Rachel  4.0     0.0


Exercise II. Run the cell below to create a new dataframe called `df_miss`.  Its first column will contain some missing values.

In [11]:
import pandas as pd
import numpy as np
import random

nrows = 10
ncols = 5

# set a seed for random number generation
np.random.seed(314)
# create an array filled with random data
data = np.array(np.random.rand(nrows, ncols))
# put the data to a pandas dataframe
df_miss = pd.DataFrame(data)
# rename the columns
df_miss.columns = ['col_'+str(ii) for ii in range(ncols)]

# randomly set some values to missing
ix0 = np.random.randint(nrows, size=3)
ix1 = np.random.randint(nrows, size=3)

df_miss['col_0'][ix0] = np.nan
df_miss['col_1'][ix1] = np.nan

print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0       NaN       NaN  0.265048  0.783205  0.918001
1  0.827355       NaN  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3       NaN       NaN  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5       NaN  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


/tmp/ipykernel_5654/310065607.py:21: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_miss['col_0'][ix0] = np.nan
/tmp/ipykernel_5654/310065607.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are settin

In [12]:
df_miss.loc[ix0, 'col_0'] = np.nan
df_miss.loc[ix1, 'col_1'] = np.nan

In [13]:
df_miss.fillna({"col_1": 0}, inplace=True)
# fills the NaN values in col_1 with 0
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0       NaN  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3       NaN  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5       NaN  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


In [14]:
df_miss.fillna({"col_0": df_miss["col_0"].median()}, inplace=True)
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


Impute the missing values (NaN) in `col_0` (but not `col_1`) with the median.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [15]:
df_miss.fillna({"col_0": df_miss["col_0"].median()}, inplace=True)
# fills only the NaN values in col_0 with the median of col_0
# inplace=True stores the result in df_miss itself
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


Impute the missing values in `col_1` with value 0.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [16]:
df_miss.fillna({"col_1": 0}, inplace=True)
# fills the NaN values in col_1 with the value 0
# inplace=True stores the result in df_miss itself
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720
